# Module 7.1 — Performance Profiling & Optimisation

### Python Mastery · Track 7: Performance, Internals & C Extensions

Making code faster begins with measuring, not guessing. This module covers profiling where time is spent (`cProfile`), measuring where memory goes (`tracemalloc`), benchmarking with `timeit`, and a disciplined approach to optimising the parts that actually matter.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it (`Shift + Enter`).
- Profiling and memory tools run directly in cells. Line-level profiling uses a script run with `!kernprof`, since it instruments a decorated function.
- Attempt the exercises before consulting the solutions at the end.

### Learning objectives

After completing this module you will be able to:

1. Profile CPU time with `cProfile` and read `pstats` output.
2. Profile memory with `tracemalloc` and find allocation hot spots.
3. Use line-level profiling to pinpoint slow lines within a function.
4. Benchmark alternatives with `timeit` and compare fairly.
5. Apply a measure-first optimisation method to a realistic example.

**Prerequisites:** Tracks 1 to 6 (this extends Module 6.2).

---

## Part 1 · CPU Profiling with `cProfile`

Module 6.2 introduced `cProfile` for debugging; here we use it for systematic optimisation. The profiler records how many times each function is called and how long it takes, distinguishing **total time** (in the function itself) from **cumulative time** (including everything it calls). The bottleneck is wherever the time actually concentrates, which is rarely where intuition suggests.

In [ ]:
import cProfile, pstats, io

def transform(n):
    # An intentionally wasteful implementation: repeated work.
    return sum(i * i for i in range(n))

def build_report(values):
    result = []
    for v in values:
        result.append(transform(v))      # the heavy call, made many times
    return result

def summarise(report):
    return sum(report) / len(report)

def run():
    values = [5000] * 50
    report = build_report(values)
    return summarise(report)

profiler = cProfile.Profile()
profiler.enable()
run()
profiler.disable()

buffer = io.StringIO()
stats = pstats.Stats(profiler, stream=buffer).sort_stats("cumulative")
stats.print_stats(6)
print(buffer.getvalue())

The report makes the structure clear: `run` has high cumulative time but negligible time of its own, because it merely calls `build_report`, which in turn spends nearly all its time in `transform`. Sorting by cumulative time shows the call chain; sorting by total time (`tottime`) points to the function doing the actual work. That is the function to optimise.

## Part 2 · Memory Profiling with `tracemalloc`

Time is only one resource; memory matters too, especially for large data. The standard `tracemalloc` module records where memory is allocated. You start tracing, run the code, then ask for the top allocation sites. This reveals which lines are responsible for the most memory, the memory equivalent of a CPU hot spot.

In [ ]:
import tracemalloc

def make_data():
    # Several allocations of different sizes.
    small = [0] * 1000
    medium = [list(range(100)) for _ in range(100)]   # a list of lists
    big = list(range(500_000))                         # the largest allocation
    return small, medium, big

tracemalloc.start()
data = make_data()

# Take a snapshot and report the largest allocations by source line.
snapshot = tracemalloc.take_snapshot()
top_stats = snapshot.statistics("lineno")
print("top allocation sites:")
for stat in top_stats[:3]:
    print("  ", stat)

current, peak = tracemalloc.get_traced_memory()
print(f"current traced memory: {current/1024:.1f} KiB")
print(f"peak traced memory:    {peak/1024:.1f} KiB")
tracemalloc.stop()

`tracemalloc` can also **compare** two snapshots to find what grew between them, which is the standard technique for hunting memory leaks: take a snapshot, run an operation that should release everything, take another, and look at the difference.

In [ ]:
import tracemalloc

tracemalloc.start()
baseline = tracemalloc.take_snapshot()

# Simulate work that retains memory unexpectedly.
retained = []
for _ in range(100):
    retained.append([0] * 1000)          # this keeps growing; a leak-like pattern

after = tracemalloc.take_snapshot()
diff = after.compare_to(baseline, "lineno")
print("largest growth between snapshots:")
for stat in diff[:2]:
    print("  ", stat)
tracemalloc.stop()

## Part 3 · Line-Level Profiling

`cProfile` works at function granularity. To see which **lines** within a hot function are slow, use a line profiler. The `line_profiler` package provides a `@profile` decorator and a `kernprof` runner. Because it instruments a decorated function in a script, the example writes a small file and runs it with `!kernprof`.

In [ ]:
%%writefile line_demo.py
@profile                                  # line_profiler injects this name when run via kernprof
def compute():
    total = 0
    for i in range(100000):               # a cheap loop
        total += i
    squares = [x * x for x in range(100000)]   # a heavier line
    biggest = max(squares)                # another pass over the data
    return total, biggest

if __name__ == "__main__":
    compute()

In [ ]:
# kernprof runs the script and records per-line timings; -l line mode, -v prints the report.
!kernprof -l -v line_demo.py

The per-line report shows the percentage of time on each line, so you can see, for example, that building the list of squares and the `max` pass dominate, while the simple accumulation loop is comparatively cheap. This is the finest-grained view before dropping to the bytecode level (Module 7.4).

## Part 4 · Benchmarking with `timeit`

To compare specific alternatives fairly, `timeit` runs each many times and reports the total, smoothing out noise. Use it to settle concrete questions, but measure realistic inputs and be sceptical of tiny differences, which rarely matter in practice.

In [ ]:
import timeit

# Question: for membership tests, how do a list and a set compare?
setup_list = "data = list(range(10000))"
setup_set = "data = set(range(10000))"
stmt = "9999 in data"          # worst case for a list: the value is at the end

list_time = timeit.timeit(stmt, setup=setup_list, number=10000)
set_time = timeit.timeit(stmt, setup=setup_set, number=10000)

print(f"list membership: {list_time:.4f}s")
print(f"set membership:  {set_time:.5f}s")
print(f"the set is roughly {list_time/set_time:.0f}x faster here")
print("because a set uses hashing (O(1)) while a list scans (O(n))")

## Part 5 · A Disciplined Optimisation: Measure, Change, Re-measure

The method matters more than any single trick. Profile to find the real bottleneck, make one targeted change, then re-measure to confirm it helped, and stop once the code is fast enough. The example below takes a slow function, identifies the cause, applies the right fix, and verifies the gain, all while keeping the output identical.

In [ ]:
import timeit

# Slow version: repeatedly searching a list inside a loop (O(n*m)).
def count_common_slow(items, allowed):
    count = 0
    for item in items:
        if item in allowed:           # 'allowed' is a list -> linear scan each time
            count += 1
    return count

# Fast version: convert the lookup collection to a set once (O(n)).
def count_common_fast(items, allowed):
    allowed_set = set(allowed)        # one-time conversion
    return sum(1 for item in items if item in allowed_set)

items = list(range(2000))
allowed = list(range(1000, 3000))

# Confirm identical behaviour before trusting the speed-up.
assert count_common_slow(items, allowed) == count_common_fast(items, allowed)

slow = timeit.timeit(lambda: count_common_slow(items, allowed), number=200)
fast = timeit.timeit(lambda: count_common_fast(items, allowed), number=200)
print(f"slow (list lookup): {slow:.4f}s")
print(f"fast (set lookup):  {fast:.4f}s")
print(f"speed-up: about {slow/fast:.0f}x, with identical results")

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Profiling to find the slow function (Easy)

In [ ]:
import cProfile, pstats, io

def quick(): return sum(range(100))
def slow(): return sum(i * i for i in range(200_000))

def main():
    quick()
    slow()
    quick()

p = cProfile.Profile(); p.enable(); main(); p.disable()
buf = io.StringIO()
pstats.Stats(p, stream=buf).sort_stats("tottime").print_stats(3)
print(buf.getvalue())

### Example 2 — Peak memory of an operation (Easy)

In [ ]:
import tracemalloc

tracemalloc.start()
data = [list(range(1000)) for _ in range(200)]
current, peak = tracemalloc.get_traced_memory()
print(f"peak memory: {peak/1024:.1f} KiB")
tracemalloc.stop()

### Example 3 — Comparing two implementations (Medium)

In [ ]:
import timeit

# Building a string: concatenation in a loop versus str.join.
def concat():
    s = ""
    for i in range(1000):
        s += str(i)
    return s

def join():
    return "".join(str(i) for i in range(1000))

assert concat() == join()
print(f"concatenation: {timeit.timeit(concat, number=1000):.4f}s")
print(f"str.join:      {timeit.timeit(join, number=1000):.4f}s")
print("join avoids creating many intermediate strings")

### Example 4 — Finding memory growth with snapshots (Medium)

In [ ]:
import tracemalloc

tracemalloc.start()
before = tracemalloc.take_snapshot()

cache = {}
for i in range(5000):
    cache[i] = str(i) * 10           # grows steadily

after = tracemalloc.take_snapshot()
for stat in after.compare_to(before, "lineno")[:2]:
    print(stat)
tracemalloc.stop()

### Example 5 — Optimising a nested-loop hot spot (Difficult)

In [ ]:
import timeit

# Slow: recomputing a sum inside the loop.
def variance_slow(numbers):
    n = len(numbers)
    result = 0
    for x in numbers:
        mean = sum(numbers) / n          # recomputed every iteration -> O(n^2)
        result += (x - mean) ** 2
    return result / n

# Fast: compute the mean once.
def variance_fast(numbers):
    n = len(numbers)
    mean = sum(numbers) / n              # computed a single time -> O(n)
    return sum((x - mean) ** 2 for x in numbers) / n

data = list(range(2000))
assert abs(variance_slow(data) - variance_fast(data)) < 1e-6

slow = timeit.timeit(lambda: variance_slow(data), number=20)
fast = timeit.timeit(lambda: variance_fast(data), number=20)
print(f"slow: {slow:.4f}s")
print(f"fast: {fast:.4f}s")
print(f"about {slow/fast:.0f}x faster by hoisting the invariant computation")

### Example 6 — A reusable profiling helper (Difficult)

In [ ]:
import cProfile, pstats, io
from contextlib import contextmanager

@contextmanager
def profiled(top=5, sort="cumulative"):
    """Profile the enclosed block and print the top functions."""
    profiler = cProfile.Profile()
    profiler.enable()
    try:
        yield
    finally:
        profiler.disable()
        buf = io.StringIO()
        pstats.Stats(profiler, stream=buf).sort_stats(sort).print_stats(top)
        print(buf.getvalue())

def work():
    return [sum(range(i)) for i in range(2000)]

with profiled(top=3):
    work()

---

## Exercises

**Exercise 1 (Easy).** Use `cProfile` to profile a function that calls a cheap helper and an expensive helper, printing the top three functions sorted by total time.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Use `tracemalloc` to measure the peak memory used while building a list of 100,000 integers, and print it in KiB.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Use `timeit` to compare summing with a `for` loop accumulator versus the built-in `sum`, over `range(5000)`, and print both timings.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Take two `tracemalloc` snapshots around a loop that appends 2,000 lists of 500 zeros each, and print the two largest growth sites from the comparison.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** A function checks, for each word in a list, whether it appears in a second large list (using `in` on the list). Rewrite it to use a set for the lookup, confirm identical results with an assertion, and compare the timings.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import cProfile, pstats, io

def cheap(): return sum(range(50))
def expensive(): return sum(i * i for i in range(300_000))
def main(): cheap(); expensive()

p = cProfile.Profile(); p.enable(); main(); p.disable()
buf = io.StringIO()
pstats.Stats(p, stream=buf).sort_stats("tottime").print_stats(3)
print(buf.getvalue())

**Solution 2**

In [ ]:
import tracemalloc
tracemalloc.start()
data = list(range(100_000))
_, peak = tracemalloc.get_traced_memory()
print(f"peak: {peak/1024:.1f} KiB")
tracemalloc.stop()

**Solution 3**

In [ ]:
import timeit
loop = timeit.timeit("""
total = 0
for i in range(5000):
    total += i
""", number=5000)
builtin = timeit.timeit("total = sum(range(5000))", number=5000)
print(f"loop:    {loop:.4f}s")
print(f"builtin: {builtin:.4f}s")

**Solution 4**

In [ ]:
import tracemalloc
tracemalloc.start()
before = tracemalloc.take_snapshot()
store = []
for _ in range(2000):
    store.append([0] * 500)
after = tracemalloc.take_snapshot()
for stat in after.compare_to(before, "lineno")[:2]:
    print(stat)
tracemalloc.stop()

**Solution 5**

In [ ]:
import timeit

def count_slow(words, vocabulary):
    return sum(1 for w in words if w in vocabulary)   # vocabulary is a list

def count_fast(words, vocabulary):
    vocab = set(vocabulary)
    return sum(1 for w in words if w in vocab)

words = [str(i) for i in range(2000)]
vocabulary = [str(i) for i in range(1000, 4000)]
assert count_slow(words, vocabulary) == count_fast(words, vocabulary)

slow = timeit.timeit(lambda: count_slow(words, vocabulary), number=100)
fast = timeit.timeit(lambda: count_fast(words, vocabulary), number=100)
print(f"slow: {slow:.4f}s | fast: {fast:.4f}s | speed-up about {slow/fast:.0f}x")

---

## Summary and Key Points

- Optimise by measuring first: `cProfile` reports calls and time per function, with total time (the function itself) versus cumulative time (including callees); the bottleneck is where time concentrates, not where intuition guesses.
- `tracemalloc` finds memory hot spots by source line and, by comparing two snapshots, reveals growth between them, the standard way to hunt leaks.
- Line-level profiling (`line_profiler` via `kernprof`) pinpoints slow lines within a hot function, a finer view than `cProfile`.
- `timeit` benchmarks specific alternatives fairly by running them many times; measure realistic inputs and ignore trivial differences.
- The discipline is measure, make one targeted change, re-measure, and stop when fast enough, always confirming the output is unchanged; algorithmic fixes (such as a set for membership) usually beat micro-optimisations.

### Next module: 7.2 — Memory Management

The next module looks under the hood at how Python manages memory: reference counting, the cyclic garbage collector, and techniques to reduce memory with `__slots__`, `weakref`, generators, and `array.array`.